In [1]:
import pandas as pd
import numpy as np

In [8]:
model_df = pd.read_csv('final_train_dataset_2019_2024.csv')

## Creating mapping for Tracks and Teams

In [5]:
track_dict = {track: i for i, track in enumerate(model_df['Track'].unique())}

track_dict

{'Australia': 0,
 'Bahrain': 1,
 'China': 2,
 'Azerbaijan': 3,
 'Spain': 4,
 'Monaco': 5,
 'Canada': 6,
 'France': 7,
 'Austria': 8,
 'Great Britain': 9,
 'Germany': 10,
 'Hungary': 11,
 'Belgium': 12,
 'Italy': 13,
 'Singapore': 14,
 'Russia': 15,
 'Japan': 16,
 'Mexico': 17,
 'United States': 18,
 'Brazil': 19,
 'Abu Dhabi': 20,
 nan: 21,
 'Saudi Arabia': 22,
 'Miami': 23,
 'Emilia Romagna': 24,
 'Netherlands': 25,
 'Las Vegas': 26,
 'Qatar': 27}

In [6]:
team_dict = {team: i for i, team in enumerate(model_df['Team'].unique())}

team_dict

{'Mercedes': 0,
 'Red Bull Racing': 1,
 'Ferrari': 2,
 'Haas': 3,
 'Renault': 4,
 'Alfa Romeo': 5,
 'Racing Point': 6,
 'Toro Rosso': 7,
 'McLaren': 8,
 'Williams': 9,
 'AlphaTauri': 10,
 'Aston Martin': 11,
 'Alpine': 12,
 'Haas F1 Team': 13,
 'Kick Sauber': 14,
 'RB': 15}

In [9]:
model_df['Track'] = model_df['Track'].map(track_dict)
model_df['Team'] = model_df['Team'].map(team_dict)

In [10]:
model_df.head()

,Position,Starting Grid,kaggle_driver_id,raceId,finished,dnf,laps_down,time_gap_sec,year,Team,Track,rain,sunny,quali_position,participated_q2,participated_q3,q1_time_sec,q2_time_sec,q3_time_sec
0,1,2.0,822,1010,1.0,0.0,0.0,0.000,2019,0,0,0,1,2.0,1.0,1.0,82.367,81.193,80.598
1,2,1.0,1,1010,1.0,0.0,0.0,20.886,2019,0,0,0,1,1.0,1.0,1.0,82.043,81.014,80.486
2,3,4.0,830,1010,1.0,0.0,0.0,22.520,2019,1,0,0,1,4.0,1.0,1.0,82.876,81.678,81.320
3,4,3.0,20,1010,1.0,0.0,0.0,57.109,2019,2,0,0,1,3.0,1.0,1.0,82.885,81.912,81.190
4,5,5.0,844,1010,1.0,0.0,0.0,58.230,2019,2,0,0,1,5.0,1.0,1.0,82.017,81.739,81.442


In [11]:
import json

with open("track_dict.json", "w") as f:
    json.dump(track_dict, f)

with open("team_dict.json", "w") as f:
    json.dump(team_dict, f)

In [36]:
#these columns were not numeric

In [21]:
cols_to_fix = [
    "quali_position",
    "q1_time_sec",
    "q2_time_sec",
    "q3_time_sec"
]

for col in cols_to_fix:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

## Baseline model so we will use random forest, train on 2019-2023 dataset and test on 2024 dataset

In [22]:
train_data = model_df[model_df["year"] <= 2023]
test_data  = model_df[model_df["year"] == 2024]

X_train = train_data.drop(columns=["Position","raceId","kaggle_driver_id"])
y_train = train_data["Position"]

X_test = test_data.drop(columns=["Position","raceId","kaggle_driver_id"])
y_test = test_data["Position"]

In [28]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,n_estimators,300
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:
print(X_train.dtypes)

Starting Grid      float64
finished           float64
dnf                float64
laps_down          float64
time_gap_sec       float64
year                 int64
Team                 int64
Track                int64
rain                 int64
sunny                int64
quali_position     float64
participated_q2    float64
participated_q3    float64
q1_time_sec        float64
q2_time_sec        float64
q3_time_sec        float64
dtype: object


In [37]:
# changing position as numeric and giving dnf/dsq as 21

In [25]:
model_df["Position"] = pd.to_numeric(model_df["Position"], errors="coerce")
model_df["Position"] = model_df["Position"].fillna(21).astype(int)

In [27]:
train_data = model_df[model_df["year"] <= 2023]
test_data  = model_df[model_df["year"] == 2024]

X_train = train_data.drop(columns=["Position","raceId","kaggle_driver_id"])
y_train = train_data["Position"]

X_test = test_data.drop(columns=["Position","raceId","kaggle_driver_id"])
y_test = test_data["Position"]

In [38]:
# did not exclude some columns that is why mae is comming so low

In [30]:
from sklearn.metrics import mean_absolute_error

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)

print("Test MAE (2024):", mae)

Test MAE (2024): 1.8018368055555556


In [31]:
print(X_train.columns)

Index(['Starting Grid', 'finished', 'dnf', 'laps_down', 'time_gap_sec', 'year',
       'Team', 'Track', 'rain', 'sunny', 'quali_position', 'participated_q2',
       'participated_q3', 'q1_time_sec', 'q2_time_sec', 'q3_time_sec'],
      dtype='object')


In [39]:
# removing those columns

In [32]:
X_train = train_data.drop(columns=[
    "Position",
    "raceId",
    "kaggle_driver_id",
    "finished",
    "dnf",
    "laps_down",
    "time_gap_sec"
])

y_train = train_data["Position"]

X_test = test_data.drop(columns=[
    "Position",
    "raceId",
    "kaggle_driver_id",
    "finished",
    "dnf",
    "laps_down",
    "time_gap_sec"
])

y_test = test_data["Position"]

In [33]:
model.fit(X_train, y_train)
preds = model.predict(X_test)

In [34]:
from sklearn.metrics import mean_absolute_error
print("Test MAE:", mean_absolute_error(y_test, preds))

Test MAE: 3.567878158068783


In [41]:
# dependencies of baseline models on columns

In [35]:
import pandas as pd

importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance)

quali_position     0.332042
q1_time_sec        0.142362
Starting Grid      0.118852
Team               0.117831
q2_time_sec        0.105187
q3_time_sec        0.088303
year               0.044496
Track              0.026729
rain               0.013563
sunny              0.008038
participated_q2    0.001314
participated_q3    0.001283
dtype: float64


## Adding additional features and then checking the MAE

In [43]:
model_df = model_df.sort_values(["kaggle_driver_id", "year", "raceId"])
model_df["driver_avg_finish_last5"] = (
    model_df
    .groupby("kaggle_driver_id")["Position"]
    .shift(1)                          # avoid using current race
    .rolling(window=5, min_periods=1)
    .mean()
)
model_df["driver_avg_finish_last5"] = model_df["driver_avg_finish_last5"].fillna(10)
X_train = train_data.drop(columns=[
    "Position",
    "raceId",
    "kaggle_driver_id"
])

In [45]:
train_data = model_df[model_df["year"] <= 2023]
test_data  = model_df[model_df["year"] == 2024]

In [46]:
X_train = train_data.drop(columns=[
    "Position",
    "raceId",
    "kaggle_driver_id",
    "finished",
    "dnf",
    "laps_down",
    "time_gap_sec"
])

y_train = train_data["Position"]

X_test = test_data.drop(columns=[
    "Position",
    "raceId",
    "kaggle_driver_id",
    "finished",
    "dnf",
    "laps_down",
    "time_gap_sec"
])

y_test = test_data["Position"]

In [47]:
model.fit(X_train, y_train)

preds = model.predict(X_test)

from sklearn.metrics import mean_absolute_error
print("New MAE:", mean_absolute_error(y_test, preds))

New MAE: 3.475680555555556


In [48]:
importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance)

quali_position             0.321893
driver_avg_finish_last5    0.149415
Starting Grid              0.108895
q1_time_sec                0.107085
Team                       0.089399
q2_time_sec                0.079363
q3_time_sec                0.068852
year                       0.035579
Track                      0.020611
rain                       0.010133
sunny                      0.006488
participated_q3            0.001245
participated_q2            0.001043
dtype: float64


In [49]:
model_df["grid_delta"] = model_df["Starting Grid"] - model_df["quali_position"]

In [50]:
model_df = model_df.sort_values(["Team", "year", "raceId"])
model_df["team_avg_finish_last5"] = (
    model_df
    .groupby("Team")["Position"]
    .shift(1)                # avoid using current race
    .rolling(window=5, min_periods=1)
    .mean()
)
model_df["team_avg_finish_last5"] = model_df["team_avg_finish_last5"].fillna(10)
train_data = model_df[model_df["year"] <= 2023]
test_data  = model_df[model_df["year"] == 2024]
X_train = train_data.drop(columns=[
    "Position",
    "raceId",
    "kaggle_driver_id",
    "finished",
    "dnf",
    "laps_down",
    "time_gap_sec"
])

y_train = train_data["Position"]

X_test = test_data.drop(columns=[
    "Position",
    "raceId",
    "kaggle_driver_id",
    "finished",
    "dnf",
    "laps_down",
    "time_gap_sec"
])

y_test = test_data["Position"]

In [51]:
model.fit(X_train, y_train)

preds = model.predict(X_test)

from sklearn.metrics import mean_absolute_error
print("New MAE:", mean_absolute_error(y_test, preds))

New MAE: 3.594875


In [52]:
importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance)

quali_position             0.290291
team_avg_finish_last5      0.124031
driver_avg_finish_last5    0.117458
Starting Grid              0.107597
q1_time_sec                0.082070
Team                       0.072155
q2_time_sec                0.062270
q3_time_sec                0.051297
year                       0.030457
grid_delta                 0.025409
Track                      0.018356
rain                       0.009056
sunny                      0.005972
participated_q3            0.002720
participated_q2            0.000862
dtype: float64
